# KG-Commit Package Usage Guide

This notebook demonstrates how to use the modular `kg_commit.knowledge` package to build Knowledge Graphs from commit data.

The package provides:
- **DatasetLoader**: Flexible data loading with customizable column mapping
- **CommitParser**: Regex-based extraction of code entities from diffs
- **KnowledgeGraphBuilder**: Tiered KG construction with interval tracking
- **KGActionLogger**: CSV audit trail of all KG operations

This replaces the monolithic notebook with reusable, pluggable components.

## 0 — Setup & Imports

In [5]:
from pathlib import Path
import pandas as pd
import networkx as nx
from collections import Counter, defaultdict
import time

# Import the modular KG package
from kg_commit.knowledge import (
    DatasetLoader,
    DiffTextSource,
    CommitParser,
    KnowledgeGraphBuilder,
    extract_kg_features,
    KGActionLogger,
    KGBuildMetrics,
)

print("✓ All modules imported successfully")

✓ All modules imported successfully


## 1 — Data Loading

The `DatasetLoader` provides flexible, customizable data loading with optional preprocessing.

In [11]:
# Configure paths
DATA_DIR = Path("E:/Projects/kgcommit/data/apachejit")
PROJECTS_DIR = DATA_DIR / "projects"
PROJECT_NAME = "apache/groovy"  # Match the actual project name in CSV
LOCAL_CSV = PROJECTS_DIR / "apache_groovy.csv"
DIFF_TEXT_CSV = PROJECTS_DIR / "apache_groovy_diff.csv"

print(f"Data directory: {DATA_DIR}")
print(f"Project CSV exists: {LOCAL_CSV.exists()}")
print(f"Diff CSV exists: {DIFF_TEXT_CSV.exists()}")

Data directory: E:\Projects\kgcommit\data\apachejit
Project CSV exists: True
Diff CSV exists: True


In [12]:
# ── Step 1.1: Load base commit data ──────────────────────────────────────
loader = DatasetLoader(LOCAL_CSV, format="csv")

# Preview data
print("Sample data:")
print(loader.peek(3))

# Load and normalize
df = loader.load(
    sort_by="author_date",
    filters={"project": PROJECT_NAME},
)

print(f"\nLoaded {len(df):,} commits for {PROJECT_NAME}")
print(f"Date range: {df.author_date.min()} → {df.author_date.max()}")

Sample data:
                                  commit_id        project  buggy    fix  \
0  7b8480744ea6e6fb41efd4329bb470c8f3c763db  apache/groovy  False  False   
1  192b631e7be302ecde822546ba70a9853ddbda01  apache/groovy  False  False   
2  0ab6465a7dece117c61c3efd3ec95d20524bdad6  apache/groovy  False  False   

   year  author_date   la  ld  nf  nd  ns       ent      ndev       age  \
0  2003   1070355653  372  23   8   3   3  2.669743  0.250000  3.625000   
1  2003   1063298262    2   2   2   2   1  1.000000  1.000000  0.000000   
2  2003   1069704572   41  26   3   3   2  1.237612  0.666667  5.333333   

      nuc  aexp  arexp      asexp  
0   2.125   243  243.0   0.683585  
1   2.500    19   19.0  14.000000  
2  45.000   233  233.0   0.000606  

Loaded 8,059 commits for apache/groovy
Date range: 1063292086 → 1577270991


In [13]:
# ── Step 1.2: Load diff text (optional) ──────────────────────────────────
diff_source = DiffTextSource(DIFF_TEXT_CSV, source_type="csv_file")

# Preload diff map for first N commits
DEMO_SIZE = 100  # For demonstration, use first 100 commits
commit_ids = df.iloc[:DEMO_SIZE].commit_id.tolist()
diff_map = diff_source.load_batch(commit_ids)

n_loaded = sum(1 for v in diff_map.values() if v)
print(f"Loaded diff text for {n_loaded} / {len(commit_ids)} commits")

# Show sample diff
sample_cid = commit_ids[0]
sample_diff = diff_map.get(sample_cid, "")
if sample_diff:
    print(f"\nSample diff (first 300 chars):")
    print(sample_diff[:300] + "...")

Loaded diff text for 100 / 100 commits

Sample diff (first 300 chars):
commit a322a829a586cf03685152f06cc7025d66062202
Author: Boc McWhirter <bob@codehaus.org>
Date:   2003-09-11T14:54:46+00:00

    lexing and ignoring single- and double-line comments.
    
    
    git-svn-id: http://svn.codehaus.org/groovy/trunk/groovy/groovy-core@36 a5544e8c-8a19-0410-ba12-f9...


## 2 — Parsing Commits

The `CommitParser` extracts structured information from diff text using regex patterns.

In [21]:
# ── Step 2.1: Initialize parser and show complete diff + parsing ────────
import random

parser = CommitParser()

# Find a commit with actual diff data
diffs_available = [(cid, diff) for cid, diff in diff_map.items() if diff and len(diff) > 100]
print(f"Commits with substantial diffs: {len(diffs_available)}")

if diffs_available:
    # Pick random commit with diff
    random_cid, full_diff = random.choice(diffs_available)
    print(f"\n{'='*80}")
    print(f"RANDOM COMMIT: {random_cid}")
    print(f"{'='*80}\n")
    
    print("FULL UNIFIED DIFF:")
    print("-" * 80)
    print(full_diff)
    print("-" * 80)
    
    # Parse it
    parsed = parser.parse(full_diff)
    
    print("\n\n" + "="*80)
    print("COMPLETE PARSING RESULTS")
    print("="*80)
    
    print(f"\nAuthor: {parsed['author']}")
    print(f"Message: {parsed['message'][:100] if parsed['message'] else 'N/A'}...")
    
    print(f"\nFiles ({len(parsed['files'])} total):")
    for f in parsed['files']:
        print(f"  - {f}")
    
    print(f"\nImports ({len(parsed['imports'])} total):")
    for imp in parsed['imports']:
        print(f"  - {imp}")
    
    print(f"\nClasses ({len(parsed['classes'])} total):")
    for cls in parsed['classes']:
        print(f"  - {cls}")
    
    print(f"\nFunctions ({len(parsed['functions'])} total):")
    for func in parsed['functions']:
        print(f"  - {func}")
    
    print(f"\nVariables ({len(parsed['variables'])} total):")
    for var in parsed['variables']:
        print(f"  - {var}")
    
    print(f"\nIssues ({len(parsed['issues'])} total):")
    for issue in parsed['issues']:
        print(f"  - {issue}")
else:
    print("⚠ No diffs with substantial content found!")

Commits with substantial diffs: 100

RANDOM COMMIT: 8dcc001e4e0cdc059daa0fd2e1ebcc1bd30bf222

FULL UNIFIED DIFF:
--------------------------------------------------------------------------------
commit 8dcc001e4e0cdc059daa0fd2e1ebcc1bd30bf222
Author: James Strachan <jastrachan@mac.com>
Date:   2003-09-11T19:05:46+00:00

    added try / catch / finally to the
    
    
    git-svn-id: http://svn.codehaus.org/groovy/trunk/groovy/groovy-core@59 a5544e8c-8a19-0410-ba12-f9af4593a198

--- a//dev/null
+++ b/src/main/org/codehaus/groovy/ast/CatchStatement.java
@@ -0,0 +1,81 @@
+/*
+ $Id$
+
+ Copyright 2003 (C) The Codehaus. All Rights Reserved.
+
+ Redistribution and use of this software and associated documentation
+ ("Software"), with or without modification, are permitted provided
+ that the following conditions are met:
+
+ 1. Redistributions of source code must retain copyright
+    statements and notices.  Redistributions must also contain a
+    copy of this document.
+
+ 2. Redistributi

In [20]:
# ── Step 2.2: Debug and identify parser bugs ────────────────────────────
print("="*80)
print("PARSER BUGS IDENTIFIED AND ANALYSIS")
print("="*80)

print("\n1. AUTHOR BUG: Returns tuple instead of string")
print(f"   Current: {repr(parsed['author'])}")
print(f"   Should be: 'James Strachan <jastrachan@mac.com>'")

print("\n2. MESSAGE BUG: Not capturing commit message")
print(f"   Current: {repr(parsed['message'])}")
print(f"   Should be: 'working assertions'")

print("\n3. IMPORTS BUG: Only extracting first part of package name")
print(f"   Current imports: {parsed['imports']}")
print(f"   Should be: {'java.util.ArrayList', 'java.util.HashMap', 'java.util.List', 'java.util.Map'}")

print("\n4. CLASSES BUG: Returns 0 but diff contains class definitions")
print(f"   Current: {len(parsed['classes'])} classes")
print(f"   Expected: 4 classes (ClassNode, ClassGenerator, InvokerHelper, DumpClass)")
print(f"   Reason: Regex only matches ADDED lines (^\+) but class definitions are CONTEXT lines")

print("\n5. VARIABLES BUG: Including garbage from non-code sections")
print(f"   Current variables: {parsed['variables']}")
print(f"   Problem: 'STUFF' and 'OTHER' from TODO.txt file, not code variables")

print("\n" + "="*80)
print("ROOT CAUSES:")
print("="*80)
print("✗ Bug #1: parse_author() returns tuple (name, email) instead of formatting as string")
print("✗ Bug #2: parse_commit_message() skip logic broken - doesn't properly separate headers from message")
print("✗ Bug #3: parse_imports() does .split('.')[0] - loses full import paths")
print("✗ Bug #4: _RE_CLASS requires ^\+ but classes can appear in context lines")
print("✗ Bug #5: _RE_JV_VAR pattern too loose - matches pattern fragments from TODO.txt")

PARSER BUGS IDENTIFIED AND ANALYSIS

1. AUTHOR BUG: Returns tuple instead of string
   Current: ('James Strachan', 'jastrachan@mac.com')
   Should be: 'James Strachan <jastrachan@mac.com>'

2. MESSAGE BUG: Not capturing commit message
   Current: ''
   Should be: 'working assertions'

3. IMPORTS BUG: Only extracting first part of package name
   Current imports: {'java'}
   Should be: ('java.util.ArrayList', 'java.util.HashMap', 'java.util.List', 'java.util.Map')

4. CLASSES BUG: Returns 0 but diff contains class definitions
   Current: 0 classes
   Expected: 4 classes (ClassNode, ClassGenerator, InvokerHelper, DumpClass)
   Reason: Regex only matches ADDED lines (^\+) but class definitions are CONTEXT lines

5. VARIABLES BUG: Including garbage from non-code sections
   Current variables: [('STUFF', 'OTHER'), ('assertFailedMethod', 'MethodCaller'), ('methodType', 'String'), ('field', 'FieldNode'), ('name', 'String'), ('variable', 'Variable'), ('type', 'String'), ('index', 'int')]
   Pr

<>:21: SyntaxWarning: invalid escape sequence '\+'
<>:33: SyntaxWarning: invalid escape sequence '\+'
<>:21: SyntaxWarning: invalid escape sequence '\+'
<>:33: SyntaxWarning: invalid escape sequence '\+'
C:\Users\Behnam\AppData\Local\Temp\ipykernel_11544\4227714219.py:21: SyntaxWarning: invalid escape sequence '\+'
  print(f"   Reason: Regex only matches ADDED lines (^\+) but class definitions are CONTEXT lines")
C:\Users\Behnam\AppData\Local\Temp\ipykernel_11544\4227714219.py:33: SyntaxWarning: invalid escape sequence '\+'
  print("✗ Bug #4: _RE_CLASS requires ^\+ but classes can appear in context lines")


## 3 — Building the Knowledge Graph

The `KnowledgeGraphBuilder` constructs a multi-tiered graph with full audit logging.

In [ ]:
# ── Step 3.1: Initialize builder and logger ──────────────────────────────
builder = KnowledgeGraphBuilder(
    enable_tiers=[1, 2, 3, 4]  # Enable core, author, file, and within-file tiers
)

# Optional: Create action logger
OUTPUT_DIR = Path("../../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg_actions.csv"

logger = KGActionLogger(log_path)
metrics = KGBuildMetrics()

print(f"✓ Builder initialized")
print(f"✓ Logger will write to: {log_path}")

In [ ]:
# ── Step 3.2: Build KG incrementally (simulating online learning) ────────
df_subset = df.iloc[:DEMO_SIZE].copy()
commits_data = df_subset.to_dict('records')

# Build graph incrementally to show progression
G = nx.MultiDiGraph()
prev_cid = None

print("Building knowledge graph...")
t0 = time.perf_counter()

for i, commit_data in enumerate(commits_data):
    if i % 20 == 0:
        print(f"  {i+1} / {len(commits_data)} commits processed")
    
    diff = diff_map.get(commit_data.get("commit_id"))
    
    # Log action start
    logger.log_commit_start(
        commit_data["commit_id"],
        graph_state={"n_nodes": G.number_of_nodes(), "n_edges": G.number_of_edges()},
    )
    
    # Add to graph (method shown below)
    parsed = builder.parser.parse(diff)
    cid = f"commit:{commit_data['commit_id']}"
    ts = float(commit_data.get("author_date", 0))
    
    # Simplified: just add core commit node
    if not G.has_node(cid):
        G.add_node(
            cid,
            type="COMMIT",
            commit_id=commit_data["commit_id"],
            project=commit_data.get("project", "unknown"),
            author_date=ts,
        )
        logger.log_node_add(
            commit_data["commit_id"],
            cid,
            "COMMIT",
            tier=1,
            attributes={"author_date": ts},
        )
    
    # Add precedence edge
    if prev_cid:
        G.add_edge(prev_cid, cid, rel="precedes")
        logger.log_edge_add(
            commit_data["commit_id"],
            prev_cid,
            cid,
            "precedes",
            tier=1,
        )
    
    # Track metrics
    elapsed_ms = (time.perf_counter() - t0) * 1000
    metrics.snapshot(
        commit_data["commit_id"],
        i + 1,
        G.number_of_nodes(),
        G.number_of_edges(),
        Counter(nx.get_node_attributes(G, "type").values()),
        elapsed_ms,
    )
    
    logger.log_commit_end(
        commit_data["commit_id"],
        graph_state={"n_nodes": G.number_of_nodes(), "n_edges": G.number_of_edges()},
    )
    
    prev_cid = cid

elapsed = time.perf_counter() - t0
print(f"\n✓ KG built in {elapsed:.2f}s")
print(f"  Nodes: {G.number_of_nodes():,}")
print(f"  Edges: {G.number_of_edges():,}")

## 4 — Graph Analysis & Inspection

In [ ]:
# ── Node type distribution ────────────────────────────────────────────────
node_types = Counter(d.get("type", "?") for _, d in G.nodes(data=True))
edge_rels = Counter(d.get("rel", "?") for _, _, d in G.edges(data=True))

print("Node type distribution:")
for node_type, count in sorted(node_types.items(), key=lambda x: -x[1]):
    print(f"  {node_type}: {count}")

print(f"\nEdge relation types:")
for rel_type, count in sorted(edge_rels.items(), key=lambda x: -x[1])[:5]:
    print(f"  {rel_type}: {count}")

In [ ]:
# ── Connectivity statistics ──────────────────────────────────────────────
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
avg_degree = n_edges / n_nodes if n_nodes > 0 else 0
density = nx.density(G)
n_wcc = nx.number_weakly_connected_components(G)

print("Graph statistics:")
print(f"  Nodes: {n_nodes:,}")
print(f"  Edges: {n_edges:,}")
print(f"  Avg degree: {avg_degree:.2f}")
print(f"  Density: {density:.2e}")
print(f"  Weakly connected components: {n_wcc}")

## 5 — Action Logging & Audit Trail

In [ ]:
# ── Export action log ─────────────────────────────────────────────────────
log_summary = logger.summary()
print("Logged actions summary:")
print(f"  Total actions: {log_summary.get('total_actions', 0):,}")
print(f"  Commits processed: {log_summary.get('commits_processed', 0):,}")

if "by_action_type" in log_summary:
    print(f"\n  By action type:")
    for action_type, count in sorted(
        log_summary["by_action_type"].items(), key=lambda x: -x[1]
    ):
        print(f"    {action_type}: {count}")

# Export to DataFrame
logs_df = logger.to_dataframe()
print(f"\nLogged {len(logs_df):,} action records")
print(f"CSV written to: {log_path}")

In [ ]:
# ── Preview audit trail ──────────────────────────────────────────────────
if len(logs_df) > 0:
    print("Sample audit trail (first 5 rows):")
    print(logs_df.head(5)[["timestamp", "action_type", "entity_type", "entity_id", "relation_type"]])

    print(f"\nAction type breakdown:")
    print(logs_df["action_type"].value_counts())

## 6 — Building Metrics

In [ ]:
# ── Metrics summary ──────────────────────────────────────────────────────
metrics_summary = metrics.summary()
print("Build metrics:")
print(f"  Total snapshots: {metrics_summary.get('total_snapshots', 0)}")

if "final_state" in metrics_summary:
    final = metrics_summary["final_state"]
    print(f"\n  Final state:")
    print(f"    Nodes: {final.get('n_nodes', 0):,}")
    print(f"    Edges: {final.get('n_edges', 0):,}")
    print(f"    Avg degree: {final.get('avg_degree', 0):.2f}")

# Export metrics
metrics_df = metrics.to_dataframe()
if len(metrics_df) > 0:
    print(f"\nMetrics snapshots: {len(metrics_df):,}")

## 7 — Graph Export

In [ ]:
# ── Sanitize and export graph ────────────────────────────────────────────
# Prepare for export: handle sets, lists, and None values
G_export = G.copy()
for n, d in G_export.nodes(data=True):
    d.pop("id", None)  # Remove reserved attribute
    for k, v in list(d.items()):
        if v is None:
            d[k] = ""
        elif isinstance(v, (set, list, tuple)):
            d[k] = ",".join(sorted(str(x) for x in v))

# Export to GraphML
graphml_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg_demo.graphml"
nx.write_graphml(G_export, graphml_path)
print(f"✓ GraphML exported: {graphml_path}")

# Export to JSON
import json
json_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg_demo.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(nx.node_link_data(G_export), f, ensure_ascii=False, indent=2)
print(f"✓ JSON exported: {json_path}")

## 8 — Feature Extraction for ML

In [ ]:
# ── Extract KG features for online learning ──────────────────────────────
sample_commits = commits_data[10:15]
features_list = []

for commit_data in sample_commits:
    diff = diff_map.get(commit_data["commit_id"])
    parsed = builder.parser.parse(diff)
    
    features = extract_kg_features(G, commit_data, parsed)
    features["commit_id"] = commit_data["commit_id"]
    features_list.append(features)

features_df = pd.DataFrame(features_list)
print("Extracted KG features (first 5 commits):")
print(features_df.to_string())

## 9 — Customization Examples

In [ ]:
# ── Example 1: Filter commits by date range ─────────────────────────────
from datetime import datetime, timedelta

df_filtered = df.iloc[:DEMO_SIZE].copy()
# Filter example: keep only first 1000 seconds
min_date = df_filtered["author_date"].min()
max_date = min_date + 1000  # 1000 seconds window

df_window = df_filtered[(df_filtered["author_date"] >= min_date) & 
                        (df_filtered["author_date"] <= max_date)]

print(f"Original: {len(df_filtered):,} commits")
print(f"Filtered window: {len(df_window):,} commits")

In [ ]:
# ── Example 2: Custom tier selection ────────────────────────────────────
# Build KG with only core and author tiers (faster for large datasets)
builder_minimal = KnowledgeGraphBuilder(enable_tiers=[1, 2])
print(f"✓ Minimal builder created (tiers 1-2 only)")

# Build KG with all tiers
builder_full = KnowledgeGraphBuilder(enable_tiers=[1, 2, 3, 4, 5])
print(f"✓ Full builder created (all tiers)")

print("\nThis allows trading off completeness vs performance for your use case.")

In [ ]:
# ── Example 3: Using different data sources ────────────────────────────
# Scenario 1: CSV with diff_text column
diff_from_csv_column = DiffTextSource(None, source_type="csv_column")
print("✓ CSV column diff loader configured")

# Scenario 2: Separate diff CSV file
diff_from_file = DiffTextSource(DIFF_TEXT_CSV, source_type="csv_file")
print("✓ Separate CSV file diff loader configured")

# Scenario 3: Git repository
# diff_from_git = DiffTextSource("/path/to/repo", source_type="git_repo")
print("✓ Git repository diff loader can be configured")

print("\nAll loaders follow the same interface via DiffTextSource.")

## 10 — Summary

The modular `kg_commit.knowledge` package provides:

| Module | Purpose | Key Classes |
|--------|---------|------------|
| `dataset_loader` | Flexible data loading | `DatasetLoader`, `DiffTextSource` |
| `parsers` | Code entity extraction | `CommitParser`, individual parser functions |
| `kg_builder` | Graph construction | `KnowledgeGraphBuilder`, `IntervalManager`, `extract_kg_features` |
| `kg_logger` | Action audit trail | `KGActionLogger`, `KGBuildMetrics` |

### Key Features:

✓ **Customizable loaders** - swap data sources without changing core logic
✓ **Tiered KG model** - enable only the tiers you need
✓ **Complete audit logging** - every KG operation recorded to CSV
✓ **Generalized parsers** - Python and Java/Groovy support built-in
✓ **Online learning ready** - extract features before updating graph
✓ **Production-ready** - error handling, optional dependencies, graceful fallbacks

### Next Steps:

1. Use this package in your online learning pipeline
2. Integrate with your ML model training loop
3. Extend parsers for additional languages or entity types
4. Add custom feature extractors for your specific use case